In [10]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import random
from sklearn.metrics import root_mean_squared_error

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

ratings = pd.read_csv('../data/ml-100k/u.data', sep='\t', names=['user_id', 'item_id', 'rating', 'timestamp'])
ratings_sorted = ratings.sort_values('timestamp').reset_index(drop=True)
n = len(ratings_sorted)
train_end = int(0.7 * n)
val_end = int(0.8 * n)
train_df = ratings_sorted.iloc[:train_end].copy()
val_df = ratings_sorted.iloc[train_end:val_end].copy()
test_df = ratings_sorted.iloc[val_end:].copy()
global_mean = train_df['rating'].mean()

unique_users = train_df['user_id'].unique()
unique_items = train_df['item_id'].unique()
user_to_idx = {uid: i for i, uid in enumerate(unique_users)}
item_to_idx = {iid: i for i, iid in enumerate(unique_items)}
n_users, n_items = len(unique_users), len(unique_items)

def make_tensor(df):
    u = df['user_id'].map(user_to_idx)
    i = df['item_id'].map(item_to_idx)

    mask = u.notna() & i.notna()
    return (torch.tensor(u[mask].values.astype(int), dtype=torch.long, device=device), 
            torch.tensor(i[mask].values.astype(int), dtype=torch.long, device=device), 
            torch.tensor(df['rating'][mask].values, dtype=torch.float, device=device))

train_u, train_i, train_r = make_tensor(train_df)
val_u, val_i, val_r = make_tensor(val_df)
test_u, test_i, test_r = make_tensor(test_df)

print(f'Train: {len(train_u)}, Val: {len(val_u)}, Test: {len(test_u)}')
print(f'n_users: {n_users}, n_items: {n_items}, global_mean: {global_mean:.3f}')

Train: 70000, Val: 1694, Test: 2286
n_users: 674, n_items: 1573, global_mean: 3.530


In [11]:
class NeuralCF(nn.Module):
    def __init__(self, n_users, n_items, k=32, hidden = [64, 32], dropout=0.2):
        super().__init__()

        self.user_emb = nn.Embedding(n_users, k)
        self.item_emb = nn.Embedding(n_items, k)

        layers = []
        input_dim = 2 * k

        for h in hidden:
            layers.append(nn.Linear(input_dim, h))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            input_dim = h
        layers.append(nn.Linear(input_dim, 1))
        self.mlp = nn.Sequential(*layers)

        self.global_mean = global_mean

        nn.init.normal_(self.user_emb.weight, std=0.1)
        nn.init.normal_(self.item_emb.weight, std=0.1)

    def forward(self, user, item):
        p = self.user_emb(user)
        q = self.item_emb(item)
        x = torch.cat([p, q], dim=1)
        out = self.mlp(x).squeeze()
        return out + self.global_mean

model = NeuralCF(n_users, n_items).to(device)
print(model)

NeuralCF(
  (user_emb): Embedding(674, 32)
  (item_emb): Embedding(1573, 32)
  (mlp): Sequential(
    (0): Linear(in_features=64, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=64, out_features=32, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.2, inplace=False)
    (6): Linear(in_features=32, out_features=1, bias=True)
  )
)


In [12]:
def train_ncf(k=32, hidden=[64, 32], dropout=0.2, lr=0.01, weight_decay=1e-5, batch_size=1024, patience=5, max_epoch=100, verbose=True):
    set_seed(42)
    model = NeuralCF(n_users, n_items, k=k, hidden=hidden, dropout=dropout).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.MSELoss()

    n_train = len(train_u)
    best_val_rmse = float('inf')
    best_state = None
    epochs_no_improve = 0

    for epoch in range(max_epoch):
        model.train()

        perm = torch.randperm(n_train, device=device)

        for start in range(0, n_train, batch_size):
            idx = perm[start:(start + batch_size)]
            b_u = train_u[idx]
            b_i = train_i[idx]
            b_r = train_r[idx]

            pred = model(b_u, b_i)
            loss = loss_fn(pred, b_r)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            val_rmse = loss_fn(model(val_u, val_i), val_r).item() ** 0.5
            if verbose:
                train_rmse = loss_fn(model(train_u, train_i), train_r).item() ** 0.5
        if verbose:
            print(f'Эпоха {epoch+1:3d} | Train: {train_rmse:.4f} | Val: {val_rmse:.4f}')

        if val_rmse < best_val_rmse:
            best_val_rmse = val_rmse
            epochs_no_improve = 0
            best_state = model.state_dict()
        else:
            epochs_no_improve += 1

        if epochs_no_improve >= patience:
            if verbose:
                print(f'\nEarly stop на эпохе {epoch+1}. Лучший Val RMSE: {best_val_rmse:.4f}')
            break

    model.load_state_dict(best_state)
    return best_val_rmse, model

val_rmse_ncf, ncf_model = train_ncf()
print(f'\nNeural CF Val RMSE: {val_rmse_ncf:.4f}')

Эпоха   1 | Train: 0.9168 | Val: 0.9826
Эпоха   2 | Train: 0.8997 | Val: 0.9726
Эпоха   3 | Train: 0.8928 | Val: 0.9840
Эпоха   4 | Train: 0.8727 | Val: 0.9731
Эпоха   5 | Train: 0.8625 | Val: 0.9782
Эпоха   6 | Train: 0.8402 | Val: 0.9938
Эпоха   7 | Train: 0.8323 | Val: 0.9782

Early stop на эпохе 7. Лучший Val RMSE: 0.9726

Neural CF Val RMSE: 0.9726


0.9726 уже лучше чем было у матричной факторизации на валидационной выборке. Теперь затюним модель с помощью optuna

In [15]:
import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    k = trial.suggest_categorical('k', [16, 32, 64])
    dropout = trial.suggest_float('dropout', 0.0, 0.5)
    weight_decay = trial.suggest_float('weight_decay', 1e-6, 1e-3, log=True)
    lr = trial.suggest_float('lr', 1e-4, 1e-2, log=True)
    n_layers = trial.suggest_categorical('n_layers', [1, 2])

    if n_layers == 1:
        hidden =  [trial.suggest_categorical('h1_single', [64, 128])]
    else:
        hidden = [trial.suggest_categorical('h1_first', [64, 128]) , trial.suggest_categorical('h2_second', [16, 32, 64])]

    val_rmse, _ = train_ncf(k=k, hidden=hidden, dropout=dropout, lr=lr, weight_decay=weight_decay, verbose=False)
    return val_rmse

study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=30)

print(f'\nЛучший Val RMSE: {study.best_value:.4f}')
print(f'Лучшие параметры: {study.best_params}')


Лучший Val RMSE: 0.9460
Лучшие параметры: {'k': 64, 'dropout': 0.07895670760339726, 'weight_decay': 3.653564395082524e-05, 'lr': 0.0066658860457538055, 'n_layers': 1, 'h1_single': 64}


Optuna выбрала однослойную сеть со слабой регуляризацией - подтверждает, что для рекомендаций на малом датасете глубина не нужна, важнее качество эмбеддингов (k=64). Это согласуется с известными результатами о том, что глубокие MLP редко превосходят простые методы в рекомендациях. Теперь проверим результат модели на тестовой выборке

In [18]:
best = study.best_params
if best['n_layers'] == 1:
    hidden_final = [best['h1_single']]
else:
    hidden_final = [best['h1_first'], best['h2_second']]

print(f'Финальная архитектура: {hidden_final}')
set_seed(42)
val_rmse_final, final_ncf = train_ncf(k=best['k'], hidden=hidden_final, dropout=best['dropout'], lr=best['lr'], weight_decay=best['weight_decay'], verbose=False)
print(f'Финальный Neural CF Val RMSE: {val_rmse_final:.4f}')

final_ncf.eval()

def ncf_predict_full(df):
    preds = np.full(len(df), global_mean)
    u = df['user_id'].map(user_to_idx)
    i = df['item_id'].map(item_to_idx)
    mask = (u.notna() & i.notna()).values
    if mask.sum() > 0:
        u_t = torch.tensor(u[mask].values.astype(int), dtype=torch.long, device=device)
        i_t = torch.tensor(i[mask].values.astype(int), dtype=torch.long, device=device)
        with torch.no_grad():
            preds_mask = final_ncf(u_t, i_t).cpu().numpy()
    return preds

test_preds = ncf_predict_full(test_df)
test_rmse_ncf = root_mean_squared_error(test_df['rating'].values, test_preds)
print(f'Neural CF test RMSE (весь test, с откатом): {test_rmse_ncf:.4f}')

Финальная архитектура: [64]
Финальный Neural CF Val RMSE: 0.9460
Neural CF test RMSE (весь test, с откатом): 1.1185


## Вывод по Neural Collaborative Filtering

Реализована нейросетевая рекомендательная модель на PyTorch: эмбеддинги юзера и
фильма конкатенируются и подаются в MLP со скрытыми слоями (ReLU, Dropout).
В отличие от MF (жёсткое скалярное произведение), MLP выучивает нелинейное
взаимодействие векторов.

### Методология
- **Обучение по мини-батчам** (batch_size=1024) с перемешиванием каждую эпоху.
- **Регуляризация:** Dropout + L2 (weight_decay). Работают только в train
  (model.train() / model.eval() переключают режим).
- **Early stopping** по валидации для числа эпох.
- **Тюнинг гиперпараметров через Optuna** (30 trials): k, dropout, weight_decay,
  lr, число и размер слоёв. Байесовская оптимизация, а не слепой перебор.

### Результаты (RMSE)

| Модель | Val (покрытые пары) | Test (весь test, с откатом) |
|--------|--------------------|-----------------------------|
| Item mean (baseline) | - | 1.0367 |
| MF (k=5) | 0.9846 | 1.1085 |
| Neural CF (базовый) | 0.9726 | - |
| **Neural CF (Optuna)** | **0.9460** | 1.1185 |

Лучшие параметры: k=64, 1 скрытый слой (64 нейрона), dropout≈0.08,
weight_decay≈3.7e-5, lr≈0.007.

### Ключевые наблюдения
- **Neural CF - лучшая модель на покрытых парах** (Val 0.9460), обходит и MF (0.9846),
  и baseline (1.0367). Нелинейное взаимодействие эмбеддингов дало выигрыш там,
  где модель применима.
- **Optuna выбрала 1 скрытый слой**, а не 2 - эмпирическое подтверждение, что для
  рекомендаций на малом датасете глубина не нужна. Согласуется с известными
  результатами: глубокие MLP редко превосходят простые методы в рекомендациях.
- **Слабая регуляризация** (dropout 0.08) - однослойная сеть и так не переобучается.
- **На всём тесте (1.1185) Neural CF почти не отличается от MF (1.1085)** - не потому
  что модели равны, а потому что ~80% тестовых пар это cold start, где обе
  откатываются на global_mean. Эта масса откатов доминирует в метрике и «сплющивает»
  разницу между моделями.

### Вывод
Neural CF и MF **решают sparsity** и работают лучше baseline на известных юзерах,
причём Neural CF - сильнейшая из collaborative-моделей. Но обе **не решают cold start**:
для новых юзеров/фильмов нет обученных эмбеддингов → откат на константу. При temporal
split ~80% теста это cold start, поэтому улучшение модели не отражается на общем test
RMSE. Это прямая мотивация к content-based подходу, который работает через признаки
объектов (жанр, год) и не требует истории оценок.